# Road Scene Segmentation (CamVid)

Semantic segmentation of urban road scenes using a **U-Net** with a pretrained encoder (via `segmentation_models_pytorch`), **Albumentations** augmentations, and per-class **IoU / Dice** evaluation.

## Project Overview

This notebook builds a complete semantic segmentation pipeline on the **CamVid** benchmark:
- Download the CamVid dataset from Kaggle
- Build an Albumentations-augmented paired image/mask dataset
- Fine-tune `smp.Unet` with a ResNet-34 ImageNet encoder
- Track per-class IoU and mean Dice during training
- Visualise predicted masks against ground truth on holdout images

## Learning Objectives

1. Understand the encoder-decoder (U-Net) architecture for dense prediction.
2. Apply paired image + mask augmentations without label leakage.
3. Evaluate semantic segmentation with mean IoU and Dice.
4. Interpret colour-coded segmentation maps on real driving scenes.

## Problem Statement

Assign a semantic class label (road, sky, pedestrian, building, …) to **every pixel** of a driving-scene image.

## Why This Project Matters

Pixel-level scene understanding is a core perception stack component for autonomous driving, robotics, and augmented-reality overlays.

## Dataset Overview

**CamVid** (Cambridge-driving Labelled Video Database):
- 367 training images (960×720 → resized to 320×320 for this notebook)
- 101 validation images
- 233 test images
- 32 original classes collapsed to **11** for the standard benchmark
- Format: RGB images + colour-encoded label PNGs

## Dataset Source and License Notes

Kaggle mirror: https://www.kaggle.com/datasets/carlolepelaars/camvid

Original project: http://mi.eng.cam.ac.uk/research/projects/VideoRec/CamVid/

License: research / non-commercial use.

## Environment Setup

```bash
pip install torch torchvision segmentation-models-pytorch albumentations             kagglehub matplotlib scikit-learn tqdm
```

In [1]:
import os, json, random
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from tqdm import tqdm
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

print("PyTorch :", torch.__version__)
print("smp     :", smp.__version__)
print("CUDA    :", torch.cuda.is_available())


E:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch : 2.6.0+cu124
smp     : 0.5.0
CUDA    : True


In [2]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR = Path(os.getcwd())
DATA_DIR = SAVE_DIR / "data"
ART_DIR  = SAVE_DIR / "artifacts"
CKPT_DIR = SAVE_DIR / "checkpoints"
for d in [DATA_DIR, ART_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE  = 320          # resize both axes to this
BATCH_SIZE  = 8
EPOCHS      = 20
LR          = 1e-4
ENCODER     = "resnet34"
WEIGHTS     = "imagenet"   # pretrained encoder weights

print("Device :", DEVICE)


Device : cuda


## Dataset Download and Loading

Downloaded via `kagglehub`; requires `KAGGLE_USERNAME` / `KAGGLE_KEY` env vars or a `~/.kaggle/kaggle.json` credential file.

In [3]:
import kagglehub

dataset_path = Path(kagglehub.dataset_download("carlolepelaars/camvid"))
print("CamVid root:", dataset_path)

# Locate image and label dirs
# Kaggle mirror structure: dataset_path/CamVid/
CAMVID_ROOT = dataset_path / "CamVid"
if not CAMVID_ROOT.exists():
    # some mirrors unpack directly
    candidates = [p for p in dataset_path.rglob("train") if p.is_dir()]
    CAMVID_ROOT = candidates[0].parent if candidates else dataset_path

print("Resolved CamVid root:", CAMVID_ROOT)


E:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


CamVid root: C:\Users\ahmad\.cache\kagglehub\datasets\carlolepelaars\camvid\versions\2
Resolved CamVid root: C:\Users\ahmad\.cache\kagglehub\datasets\carlolepelaars\camvid\versions\2\CamVid


In [4]:
# Inspect actual directory layout
for item in sorted(CAMVID_ROOT.iterdir()):
    sub = list(item.iterdir())[:3] if item.is_dir() else []
    print(f"  {item.name}/  ({len(list(item.iterdir())) if item.is_dir() else '─'} items)" if item.is_dir() else f"  {item.name}")


  class_dict.csv
  test/  (232 items)
  test_labels/  (232 items)
  train/  (369 items)
  train_labels/  (369 items)
  val/  (100 items)
  val_labels/  (100 items)


In [5]:
# Standard CamVid 11-class mapping (colour → class index)
# Each entry: (R, G, B) -> class_name
CAMVID_CLASSES = [
    ("Sky",          (128, 128, 128)),
    ("Building",     ( 64,   0, 192)),
    ("Pole",         (192, 192, 128)),
    ("Road",         (128,  64, 128)),
    ("Pavement",     ( 60,  40, 222)),
    ("Tree",         (128, 128,   0)),
    ("SignSymbol",   (192, 128, 128)),
    ("Fence",        ( 64,  64, 128)),
    ("Car",          ( 64,   0, 128)),
    ("Pedestrian",   ( 64,  64,   0)),
    ("Bicyclist",    (  0, 128, 192)),
]
NUM_CLASSES = len(CAMVID_CLASSES)
CLASS_NAMES = [c[0] for c in CAMVID_CLASSES]
CLASS_COLORS = np.array([c[1] for c in CAMVID_CLASSES], dtype=np.uint8)

# Build colour -> index lookup
COLOR_TO_IDX = {}
for idx, (name, rgb) in enumerate(CAMVID_CLASSES):
    COLOR_TO_IDX[rgb] = idx

print(f"Using {NUM_CLASSES} classes:", CLASS_NAMES)


Using 11 classes: ['Sky', 'Building', 'Pole', 'Road', 'Pavement', 'Tree', 'SignSymbol', 'Fence', 'Car', 'Pedestrian', 'Bicyclist']


## Data Validation Checks

In [6]:
def find_split_dirs(root: Path):
    """Return (img_dir, lbl_dir) for each split found under root."""
    splits = {}
    for split in ("train", "val", "test"):
        img_d = root / split
        lbl_d = root / f"{split}_labels"
        if img_d.exists() and lbl_d.exists():
            splits[split] = (img_d, lbl_d)
    return splits

split_dirs = find_split_dirs(CAMVID_ROOT)
assert split_dirs, "Could not find train/val/test dirs under CamVid root."

for split, (img_d, lbl_d) in split_dirs.items():
    imgs = sorted(img_d.glob("*.png"))
    lbls = sorted(lbl_d.glob("*.png"))
    print(f"  {split:5s}: {len(imgs)} images, {len(lbls)} labels")

assert "train" in split_dirs, "Missing train split"
print("Validation passed.")


  train: 369 images, 369 labels
  val  : 100 images, 100 labels
  test : 232 images, 232 labels
Validation passed.


In [7]:
def decode_mask(label_rgb: np.ndarray) -> np.ndarray:
    """Convert an RGB label image to a (H, W) integer class index map."""
    h, w = label_rgb.shape[:2]
    mask = np.full((h, w), NUM_CLASSES, dtype=np.int64)   # 'void' = NUM_CLASSES
    for idx, (_, rgb) in enumerate(CAMVID_CLASSES):
        match = np.all(label_rgb == rgb, axis=-1)
        mask[match] = idx
    return mask

def colorize_mask(mask: np.ndarray) -> np.ndarray:
    """Map class index (H, W) back to an RGB image for visualisation.""";
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for idx in range(NUM_CLASSES):
        rgb[mask == idx] = CLASS_COLORS[idx]
    return rgb


## Exploratory Data Analysis

In [8]:
train_img_dir, train_lbl_dir = split_dirs["train"]
train_imgs = sorted(train_img_dir.glob("*.png"))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i in range(4):
    img = cv2.cvtColor(cv2.imread(str(train_imgs[i])), cv2.COLOR_BGR2RGB)
    stem = train_imgs[i].stem
    # label file may have _L suffix
    for suffix in (f"{stem}_L.png", f"{stem}.png"):
        lbl_path = train_lbl_dir / suffix
        if lbl_path.exists():
            break
    lbl_rgb = cv2.cvtColor(cv2.imread(str(lbl_path)), cv2.COLOR_BGR2RGB)
    mask    = decode_mask(lbl_rgb)

    axes[0, i].imshow(img);           axes[0, i].set_title("Image");      axes[0, i].axis("off")
    axes[1, i].imshow(lbl_rgb);       axes[1, i].set_title("Label RGB");  axes[1, i].axis("off")
    axes[2, i].imshow(colorize_mask(mask)); axes[2, i].set_title("Decoded mask"); axes[2, i].axis("off")

plt.tight_layout()
plt.savefig(ART_DIR / "eda_samples.png", dpi=100, bbox_inches="tight")
plt.close()
print("Saved eda_samples.png")


Saved eda_samples.png


## Train / Validation / Test Split Strategy

We use the official CamVid splits provided by the dataset:
- **Train**: 367 images
- **Val**: 101 images
- **Test**: 233 images (held out until final evaluation)

## Preprocessing and Augmentation Strategy

- **Resize** all images and masks to 320×320.
- **Train augmentations**: HorizontalFlip, RandomBrightnessContrast, slight RandomRotate90 — applied identically to image and mask.
- **Normalise** with ImageNet mean/std (matching encoder pretraining).
- Masks carry integer class indices; no interpolation on label resize (nearest-neighbour).

In [9]:
def build_pair_list(img_dir: Path, lbl_dir: Path):
    pairs = []
    for img_path in sorted(img_dir.glob("*.png")):
        stem = img_path.stem
        for suffix in (f"{stem}_L.png", f"{stem}.png"):
            lbl_path = lbl_dir / suffix
            if lbl_path.exists():
                pairs.append((img_path, lbl_path))
                break
    return pairs


IMG_MEAN = (0.485, 0.456, 0.406)
IMG_STD  = (0.229, 0.224, 0.225)

def get_transforms(train: bool):
    ops = [A.Resize(IMAGE_SIZE, IMAGE_SIZE, interpolation=cv2.INTER_LINEAR,
                    mask_interpolation=cv2.INTER_NEAREST)]
    if train:
        ops += [
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.4),
            A.RandomRotate90(p=0.2),
        ]
    ops += [
        A.Normalize(mean=IMG_MEAN, std=IMG_STD),
        ToTensorV2(),
    ]
    return A.Compose(ops)


class CamVidDataset(Dataset):
    def __init__(self, pairs, transforms):
        self.pairs      = pairs
        self.transforms = transforms

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, lbl_path = self.pairs[idx]
        image   = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        lbl_rgb = cv2.cvtColor(cv2.imread(str(lbl_path)), cv2.COLOR_BGR2RGB)
        mask    = decode_mask(lbl_rgb).astype(np.int64)
        # void pixels (index == NUM_CLASSES) → ignore class 0 or a dedicated ignore index
        # we clamp them to 0 here; they are masked out in the loss
        void_pixels = mask == NUM_CLASSES
        mask[void_pixels] = 0

        out = self.transforms(image=image, mask=mask.astype(np.int32))
        return out["image"], out["mask"].long(), ~torch.from_numpy(void_pixels[:IMAGE_SIZE, :IMAGE_SIZE].copy())


train_pairs = build_pair_list(*split_dirs["train"])
val_pairs   = build_pair_list(*split_dirs["val"])
test_pairs  = build_pair_list(*split_dirs.get("test", split_dirs["val"]))

train_ds = CamVidDataset(train_pairs, get_transforms(train=True))
val_ds   = CamVidDataset(val_pairs,   get_transforms(train=False))
test_ds  = CamVidDataset(test_pairs,  get_transforms(train=False))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")


Train: 369, Val: 100, Test: 232


## Main Model / Workflow

We use `smp.Unet` with a **ResNet-34** encoder pretrained on ImageNet. The decoder upsamples via bilinear interpolation with skip connections from the encoder. The final head outputs `NUM_CLASSES` logit channels.

In [10]:
model = smp.Unet(
    encoder_name=ENCODER,
    encoder_weights=WEIGHTS,
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None,       # raw logits; loss handles softmax
)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters : {total_params:,}")
print(f"Encoder          : {ENCODER}")


Total parameters : 24,437,819
Encoder          : resnet34


In [11]:
ce_loss = nn.CrossEntropyLoss(ignore_index=-1)  # void pixels excluded

def iou_per_class(preds: torch.Tensor, targets: torch.Tensor, num_classes: int):
    """preds: (N, H, W) int, targets: (N, H, W) int. Returns tensor of shape (num_classes,).""";
    iou = torch.zeros(num_classes)
    for c in range(num_classes):
        pred_c   = (preds   == c)
        target_c = (targets == c)
        inter    = (pred_c & target_c).sum().float()
        union    = (pred_c | target_c).sum().float()
        iou[c]   = inter / (union + 1e-6) if union > 0 else float("nan")
    return iou

def dice_per_class(preds: torch.Tensor, targets: torch.Tensor, num_classes: int):
    dice = torch.zeros(num_classes)
    for c in range(num_classes):
        pred_c   = (preds   == c).float()
        target_c = (targets == c).float()
        inter    = (pred_c * target_c).sum()
        total    = pred_c.sum() + target_c.sum()
        dice[c]  = 2*inter / (total + 1e-6) if total > 0 else float("nan")
    return dice


## Training Loop

In [12]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_miou = 0.0
history   = []

for epoch in range(1, EPOCHS + 1):
    # ── Train ──────────────────────────────────────────────────────────
    model.train()
    train_loss = []
    for imgs, masks, _ in tqdm(train_loader, desc=f"Ep {epoch}/{EPOCHS} train", leave=False):
        imgs  = imgs.to(DEVICE)
        masks = masks.to(DEVICE)

        logits = model(imgs)
        loss   = ce_loss(logits, masks)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        train_loss.append(loss.item())

    # ── Validate ────────────────────────────────────────────────────────
    model.eval()
    val_loss, all_iou, all_dice = [], [], []
    with torch.no_grad():
        for imgs, masks, _ in val_loader:
            imgs  = imgs.to(DEVICE)
            masks = masks.to(DEVICE)
            logits = model(imgs)
            val_loss.append(ce_loss(logits, masks).item())

            preds = logits.argmax(dim=1).cpu()
            masks_cpu = masks.cpu()
            all_iou.append(iou_per_class(preds.view(-1), masks_cpu.view(-1), NUM_CLASSES))
            all_dice.append(dice_per_class(preds.view(-1), masks_cpu.view(-1), NUM_CLASSES))

    scheduler.step()

    t_loss  = float(np.mean(train_loss))
    v_loss  = float(np.mean(val_loss))
    iou_mat = torch.stack(all_iou)            # (num_batches, num_classes)
    miou    = float(torch.nanmean(iou_mat).item())
    mdice   = float(torch.nanmean(torch.stack(all_dice)).item())

    history.append({"epoch": epoch, "train_loss": t_loss, "val_loss": v_loss,
                    "miou": miou, "mdice": mdice})

    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), CKPT_DIR / "best_unet.pth")

    print(f"Ep {epoch:>2}: train={t_loss:.4f}  val={v_loss:.4f}  mIoU={miou:.4f}  mDice={mdice:.4f}")

print(f"Best mIoU: {best_miou:.4f}")


Ep 1/20 train:   0%|          | 0/47 [00:00<?, ?it/s]

Ep 1/20 train:   2%|▏         | 1/47 [00:03<02:18,  3.00s/it]

Ep 1/20 train:   4%|▍         | 2/47 [00:04<01:38,  2.19s/it]

Ep 1/20 train:   6%|▋         | 3/47 [00:06<01:26,  1.97s/it]

Ep 1/20 train:   9%|▊         | 4/47 [00:08<01:22,  1.91s/it]

...

Ep 20/20 train:  94%|█████████▎| 44/47 [01:14<00:05,  1.81s/it]

Ep 20/20 train:  96%|█████████▌| 45/47 [01:16<00:03,  1.81s/it]

Ep 20/20 train:  98%|█████████▊| 46/47 [01:18<00:01,  1.80s/it]

Ep 20/20 train: 100%|██████████| 47/47 [01:18<00:00,  1.34s/it]

Ep 20: train=0.2910  val=0.3138  mIoU=0.3604  mDice=0.4028
Best mIoU: 0.3612


In [13]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep = [h["epoch"] for h in history]
axes[0].plot(ep, [h["train_loss"] for h in history], label="Train")
axes[0].plot(ep, [h["val_loss"]   for h in history], label="Val")
axes[0].set(xlabel="Epoch", ylabel="CE Loss", title="Training Loss Curve"); axes[0].legend()

axes[1].plot(ep, [h["miou"]  for h in history], label="mIoU",  color="tab:blue")
axes[1].plot(ep, [h["mdice"] for h in history], label="mDice", color="tab:orange")
axes[1].set(xlabel="Epoch", ylabel="Score", title="mIoU and mDice"); axes[1].legend()

plt.tight_layout()
plt.savefig(ART_DIR / "training_curves.png", dpi=100, bbox_inches="tight")
plt.close()
print("Saved training_curves.png")


Saved training_curves.png


## Inference Examples

In [14]:
model.load_state_dict(torch.load(CKPT_DIR / "best_unet.pth", map_location=DEVICE))
model.eval()

vis_loader = DataLoader(test_ds, batch_size=1, shuffle=False)
examples   = []
with torch.no_grad():
    for imgs, masks, _ in vis_loader:
        logits = model(imgs.to(DEVICE))
        pred   = logits.argmax(dim=1).squeeze(0).cpu().numpy()
        examples.append((imgs[0], masks[0].numpy(), pred))
        if len(examples) >= 6:
            break

# Denormalize for display
mean_t = torch.tensor(IMG_MEAN).view(3,1,1)
std_t  = torch.tensor(IMG_STD).view(3,1,1)

fig, axes = plt.subplots(6, 3, figsize=(12, 22))
cols = ["Image", "Ground Truth", "Prediction"]
for i, (img_t, gt_mask, pred_mask) in enumerate(examples):
    img_disp = ((img_t * std_t + mean_t).permute(1,2,0).clamp(0,1).numpy() * 255).astype(np.uint8)
    axes[i, 0].imshow(img_disp);               axes[i, 0].axis("off")
    axes[i, 1].imshow(colorize_mask(gt_mask)); axes[i, 1].axis("off")
    axes[i, 2].imshow(colorize_mask(pred_mask)); axes[i, 2].axis("off")
    if i == 0:
        for j, col in enumerate(cols):
            axes[0, j].set_title(col, fontsize=12)

plt.suptitle("Image / Ground Truth / Predicted Mask", fontsize=13)
plt.tight_layout()
plt.savefig(ART_DIR / "qualitative_masks.png", dpi=120, bbox_inches="tight")
plt.close()
print("Saved qualitative_masks.png")


Saved qualitative_masks.png


## Evaluation (IoU / Dice per class)

In [15]:
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

all_iou_test  = []
all_dice_test = []

model.eval()
with torch.no_grad():
    for imgs, masks, _ in tqdm(test_loader, desc="Evaluating", leave=False):
        logits = model(imgs.to(DEVICE))
        preds  = logits.argmax(dim=1).cpu()
        all_iou_test.append(iou_per_class(preds.view(-1),  masks.view(-1), NUM_CLASSES))
        all_dice_test.append(dice_per_class(preds.view(-1), masks.view(-1), NUM_CLASSES))

iou_mat  = torch.stack(all_iou_test)   # (batches, classes)
dice_mat = torch.stack(all_dice_test)

class_iou  = torch.nanmean(iou_mat,  dim=0).numpy()
class_dice = torch.nanmean(dice_mat, dim=0).numpy()
miou_test  = float(np.nanmean(class_iou))
mdice_test = float(np.nanmean(class_dice))

print(f"\nTest  mIoU  : {miou_test:.4f}   mDice: {mdice_test:.4f}")
print(f"{'Class':<16} {'IoU':>6}  {'Dice':>6}")
print("-" * 32)
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:<16} {class_iou[i]:>6.4f}  {class_dice[i]:>6.4f}")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x = np.arange(NUM_CLASSES)
axes[0].bar(x, class_iou,  color="steelblue"); axes[0].set_xticks(x); axes[0].set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
axes[0].set(ylabel="IoU",  title=f"Per-class IoU  (mean={miou_test:.4f})")
axes[1].bar(x, class_dice, color="coral"); axes[1].set_xticks(x); axes[1].set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
axes[1].set(ylabel="Dice", title=f"Per-class Dice (mean={mdice_test:.4f})")
plt.tight_layout()
plt.savefig(ART_DIR / "per_class_metrics.png", dpi=100, bbox_inches="tight")
plt.close()
print("Saved per_class_metrics.png")


Evaluating:   0%|          | 0/29 [00:00<?, ?it/s]

Evaluating:   3%|▎         | 1/29 [00:01<00:51,  1.83s/it]

Evaluating:   7%|▋         | 2/29 [00:03<00:46,  1.71s/it]

Evaluating:  10%|█         | 3/29 [00:05<00:47,  1.83s/it]

Evaluating:  14%|█▍        | 4/29 [00:07<00:43,  1.74s/it]

...

Evaluating:  93%|█████████▎| 27/29 [00:47<00:03,  1.73s/it]

Evaluating:  97%|█████████▋| 28/29 [00:48<00:01,  1.70s/it]

Evaluating: 100%|██████████| 29/29 [00:50<00:00,  1.73s/it]


Test  mIoU  : 0.3297   mDice: 0.3713
Class               IoU    Dice
--------------------------------
Sky              0.8645  0.9267
Building         0.0000  0.0000
Pole             0.0000  0.0000
Road             0.8868  0.9390
Pavement            nan     nan
Tree             0.6590  0.7753
SignSymbol       0.0000  0.0000
Fence            0.2220  0.2942
Car              0.6647  0.7764
Pedestrian       0.0000  0.0000
Bicyclist        0.0005  0.0009


Saved per_class_metrics.png


In [16]:
metrics = {
    "dataset": "CamVid",
    "model": f"smp.Unet + {ENCODER}",
    "image_size": IMAGE_SIZE,
    "test_miou": miou_test,
    "test_mdice": mdice_test,
    "per_class": {
        name: {"iou": float(class_iou[i]), "dice": float(class_dice[i])}
        for i, name in enumerate(CLASS_NAMES)
    }
}
with open(SAVE_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))


{
  "dataset": "CamVid",
  "model": "smp.Unet + resnet34",
  "image_size": 320,
  "test_miou": 0.329745888710022,
  "test_mdice": 0.3712601065635681,
  "per_class": {
    "Sky": {
      "iou": 0.8644834756851196,
      "dice": 0.9267334938049316
    },
    "Building": {
      "iou": 0.0,
      "dice": 0.0
    },
    "Pole": {
      "iou": 7.93132494436577e-06,
      "dice": 1.585900281497743e-05
    },
    "Road": {
      "iou": 0.8867766261100769,
      "dice": 0.9389682412147522
    },
    "Pavement": {
      "iou": NaN,
      "dice": NaN
    },
    "Tree": {
      "iou": 0.658961832523346,
      "dice": 0.7752838730812073
    },
    "SignSymbol": {
      "iou": 0.0,
      "dice": 0.0
    },
    "Fence": {
      "iou": 0.22201959788799286,
      "dice": 0.2942442297935486
    },
    "Car": {
      "iou": 0.6647283434867859,
      "dice": 0.7763955593109131
    },
    "Pedestrian": {
      "iou": 1.394087666994892e-05,
      "dice": 2.7870486519532278e-05
    },
    "Bicyclist": {
   

## Error Analysis

Inspect the 4 test images with the lowest mIoU to understand failure modes.

In [17]:
per_img_miou = []
model.eval()
with torch.no_grad():
    for i in range(min(len(test_ds), 200)):
        img_t, mask_t, _ = test_ds[i]
        logit = model(img_t.unsqueeze(0).to(DEVICE))
        pred  = logit.argmax(dim=1).squeeze(0).cpu()
        img_iou = iou_per_class(pred.view(-1), mask_t.view(-1), NUM_CLASSES)
        per_img_miou.append((float(torch.nanmean(img_iou).item()), i))

worst = sorted(per_img_miou)[:4]

mean_t = torch.tensor(IMG_MEAN).view(3,1,1)
std_t  = torch.tensor(IMG_STD).view(3,1,1)

fig, axes = plt.subplots(4, 3, figsize=(12, 16))
with torch.no_grad():
    for r, (miou_val, idx) in enumerate(worst):
        img_t, mask_t, _ = test_ds[idx]
        pred = model(img_t.unsqueeze(0).to(DEVICE)).argmax(dim=1).squeeze(0).cpu().numpy()
        img_disp = ((img_t * std_t + mean_t).permute(1,2,0).clamp(0,1).numpy() * 255).astype(np.uint8)
        axes[r, 0].imshow(img_disp);                     axes[r, 0].axis("off"); axes[r, 0].set_title("Image")
        axes[r, 1].imshow(colorize_mask(mask_t.numpy())); axes[r, 1].axis("off"); axes[r, 1].set_title("GT")
        axes[r, 2].imshow(colorize_mask(pred));           axes[r, 2].axis("off"); axes[r, 2].set_title(f"Pred mIoU={miou_val:.3f}")

plt.suptitle("4 Worst Predictions (lowest mIoU)", fontsize=12)
plt.tight_layout()
plt.savefig(ART_DIR / "error_analysis.png", dpi=120, bbox_inches="tight")
plt.close()
print("Saved error_analysis.png")


Saved error_analysis.png


## Limitations

- Void pixels (unlabelled areas) are clamped to class 0 rather than properly excluded from the loss — this adds slight label noise.
- Only 11 of CamVid's 32 classes are used (standard benchmark subset).
- 320×320 crop trades spatial detail for training speed; larger inputs improve results.

## How to Improve This Project

1. Use `smp.DeepLabV3Plus` with a stronger encoder (EfficientNet-B4, ResNeXt-50).
2. Add proper void/ignore masking in both the loss and the IoU metric.
3. Increase resolution to 480×480 or use multiscale inference.
4. Apply class-frequency weighting in the cross-entropy loss to handle imbalance.

## Production Considerations

- Export via `torch.onnx.export` for deployment on edge devices (TensorRT, OpenVINO).
- Tile large frames and merge predictions with stride overlap.
- Monitor per-class IoU drift to detect domain shift.

## Common Mistakes

- Using `INTER_LINEAR` (bilinear) interpolation for the label mask (distorts class indices).
- Normalising the mask values instead of integer-encoding them.
- Forgetting `model.eval()` + `torch.no_grad()` during validation — causes BatchNorm/Dropout differences.

## Mini Challenge / Exercises

1. Replace U-Net with `smp.FPN` and compare mIoU at the same number of epochs.
2. Add a second augmentation (GridDistortion) and measure its effect on validation mIoU.
3. Visualise the per-class confusion matrix and identify the most-confused class pairs.

## Final Summary / Key Takeaways

This notebook built a full semantic segmentation pipeline on CamVid using a pretrained U-Net:

| Stage | Key choice |
|-------|-----------|
| Data | CamVid 11-class benchmark |
| Augmentation | HFlip, BrightnessContrast, Rotate90 via Albumentations |
| Model | `smp.Unet` + ResNet-34 ImageNet encoder |
| Loss | CrossEntropy (void pixels clamped) |
| Metrics | per-class IoU, per-class Dice, mIoU, mDice |
| Evaluation | Colour-coded mask overlays + bar charts |
